# Promo depth vs media — decision arms with per-arm cost bases

The optimizer's decision vector was dollars of media spend. A promotion's cost
is not a spend line — it is **margin given away** (depth × price × units) — so
"should we fund a deeper promo or more media?" had no honest answer. This
notebook walks the #226 surface:

1. the `promo_and_media` world, whose economics and optimal split are **frozen
   before any model runs**
2. `promo_roi` — margin dollars back per margin dollar given away, with its
   refusals (event flags, unknown units, missing valuation)
3. the **cost-space reduction**: every arm re-parameterized by realized cost,
   so the existing allocator (constraints, risk objectives, per-draw
   uncertainty) runs untouched
4. the joint solve, graded against the planted optimum on **true** profit
5. `price_whatif` — a price scenario that evaluates and **refuses to recommend**
6. the endogenous negative control: clearance timing attenuates the lift, and
   the extended endogeneity screen says which lever and why

The headline the epic wanted — "trade a price cut against media" — is
deliberately *not* shipped as a recommendation: the repo's own measurement
recovers 39% of a planted price elasticity, confidently. Promo depth is the
shipping headline.

In [1]:
import warnings; warnings.filterwarnings("ignore")
import os, time
os.environ.setdefault("TQDM_DISABLE", "1")

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from pathlib import Path

pio.templates.default = "plotly_white"
pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 170)

import logging
from loguru import logger
logger.disable("mmm_framework")
for _n in ("pymc", "pymc.sampling", "numpyro", "jax", "arviz", "pytensor"):
    logging.getLogger(_n).setLevel(logging.ERROR)

INK, MUTED, TRUTH = "#1f2430", "#8a8f98", "#111418"
GOOD, BAD, GOLD = "#3d7a5c", "#b4552d", "#c9962e"
PALETTE = {"TV": "#4464ad", "Search": "#c9962e", "Social": "#3d7a5c",
           "Display": "#b4552d", "Promo depth (Promo)": "#7a4bb3"}

def style(fig, height=380, title=None, **kw):
    fig.update_layout(height=height, title=title, margin=dict(t=64, l=64, r=30, b=52),
                      font=dict(size=12), **kw)
    return fig

ART = Path.cwd().parent / "artifacts" / "promo_depth"
ART.mkdir(parents=True, exist_ok=True)
print("Setup ready.")

Setup ready.


## 1 · A world with frozen economics

The answer key is planted from DGP parameters **first** and frozen in the
world's notes — the optimizer never sees the construction, and the profit
claim below is labelled *conditional on the stated economics* because the
optimizer is handed the same margin and promo cost the answer key used.
Media is near-saturated at current spend; promo is far from saturation. The
joint optimum genuinely wants promo money.

In [2]:
from mmm_framework.synth import dgp, mff

scenario = dgp.build("promo_and_media")
t = scenario.notes

print(scenario.description, "\n")
econ = {k: t[k] for k in ("gross_margin", "promo_unit_cost", "price_reference",
                          "true_promo_lift", "true_promo_alpha")}
print("frozen economics + planted effects:")
for k, v in econ.items():
    print(f"  {k:20s} {v:,.4g}")

opt, cur = t["true_optimal_split"], t["current_split"]
print(f"\nplanted OPTIMAL split : media {opt['media_cost']:,.0f} | promo {opt['promo_cost']:,.0f}"
      f"  (promo share {opt['promo_share']:.0%}, avg depth {opt['avg_promo_depth']:.3f})")
print(f"observed CURRENT split: media {cur['media_cost']:,.0f} | promo {cur['promo_cost']:,.0f}"
      f"  (promo share {cur['promo_cost']/(cur['promo_cost']+cur['media_cost']):.0%})")
print(f"planted profit gap    : {opt['profit'] - cur['profit']:,.0f} (conditional on the stated economics)")

Promo depth and price are decisions with a planted cost basis; media near-saturated, promo far from saturation, exogenous lever timing. Grades whether the optimizer can trade promo depth against media. 

frozen economics + planted effects:
  gross_margin         0.4
  promo_unit_cost      1.2e+05
  price_reference      12.03
  true_promo_lift      7,360
  true_promo_alpha     0.45

planted OPTIMAL split : media 13,030 | promo 16,105  (promo share 55%, avg depth 0.134)
observed CURRENT split: media 23,691 | promo 5,430  (promo share 19%)
planted profit gap    : 24,287 (conditional on the stated economics)


## 2 · Fit with the levers declared

`Price` and `Promo` are declared as **levers**, not linear controls: price
becomes a sign-guarded log-price elasticity, promo a lift with its own
carryover. (The discrimination test in the suite pins that a model WITHOUT the
promo lever misses the planted split by >15pp — a world that cannot fail is
decoration.)

In [3]:
from mmm_framework.agents.fitting import build_model

DATA = ART / "promo_world.csv"
mff.scenario_to_mff(scenario).to_csv(DATA, index=False)

SPEC = {
    "kpi": "Sales",
    "kpi_level": "national",
    "media_channels": [
        {"name": c, "adstock": {"type": "geometric"},
         "saturation": {"type": "logistic"}}
        for c in scenario.channels
    ],
    "control_variables": [{"name": c} for c in scenario.controls.columns],
    # The lever declarations (#138/#222): Price and Promo leave the linear
    # control block and get their own transforms + priors.
    "price": {"variable": "Price", "reference": "median"},
    "promotions": [{"variable": "Promo", "adstock_lmax": 8}],
}
t0 = time.time()
mmm = build_model(SPEC, str(DATA))
mmm.fit(draws=800, tune=800, chains=4, random_seed=42)
print(f"NUTS fit in {time.time()-t0:.0f}s | levers: {mmm.lever_names}")

  0%|          | 0/1600 [00:00<?, ?it/s]

  0%|          | 0/1600 [00:00<?, ?it/s]

  0%|          | 0/1600 [00:00<?, ?it/s]

  0%|          | 0/1600 [00:00<?, ?it/s]

NUTS fit in 36s | levers: ['Price', 'Promo']


## 3 · Promo ROI, and what it refuses

Margin dollars returned per margin dollar given away — the number a trade
planner actually argues about. It needs a valuation and a cost basis, and both
are refusable.

In [4]:
from mmm_framework.planning import promo_roi
from mmm_framework.finance import UnresolvedValueError

value_per_kpi = t["gross_margin"] * t["price_reference"]

# Refusal 1: no valuation -> never a silent $1/KPI.
try:
    promo_roi(mmm, "Promo", unit_cost=t["promo_unit_cost"])
except UnresolvedValueError as e:
    print("without a valuation:", type(e).__name__)

r = promo_roi(mmm, "Promo", unit_cost=t["promo_unit_cost"],
              value_per_kpi=value_per_kpi, value_source="world economics")
print(f"\npromo ROI: {r.roi_mean:.2f} margin-$ back per margin-$ given away "
      f"[{r.roi_lower:.2f}, {r.roi_upper:.2f}] ({int(r.interval_mass*100)}%)")
print(f"lift: {r.lift_kpi_mean:,.0f} KPI units (planted {t['true_promo_lift']:,.0f} "
      f"-> {r.lift_kpi_mean/t['true_promo_lift']:.0%} recovered)")
print(f"cost basis: avg depth {r.avg_depth:.3f} x unit cost {r.unit_cost:,.0f} = {r.realized_cost:,.0f}")
print("\ncaveat:", r.caveats[-1])

without a valuation: UnresolvedValueError



promo ROI: 5.78 margin-$ back per margin-$ given away [5.41, 6.17] (90%)
lift: 6,521 KPI units (planted 7,360 -> 89% recovered)
cost basis: avg depth 0.045 x unit cost 120,000 = 5,430

caveat: ROI is conditional on the stated economics: the unit cost (1.2e+05 per unit average depth) and the valuation (4.814/KPI unit, source: world economics) are inputs, not measurements.


In [5]:
# Refusal 2 + 3: the cost basis is refusable. A 0/1 event flag has no depth
# (no discount cost exists); a column outside [0,1] has unknown units.
class _Stub:
    lever_names = ["Promo"]
    X_levers_raw = None

for label, series in [
    ("event flag", np.array([0.0, 1.0, 0.0, 1.0, 1.0])),
    ("unknown units", np.array([0.0, 12.0, 0.0, 40.0])),
]:
    stub = _Stub(); stub.X_levers_raw = series[:, None]
    try:
        promo_roi(stub, "Promo", unit_cost=1.0, value_per_kpi=1.0)
    except ValueError as e:
        print(f"{label}: {str(e)[:110]}...")

event flag: promo lever 'Promo' is a 0/1 event flag. A flag has no depth, so no discount cost (depth x price x units) exis...
unknown units: promo lever 'Promo' takes values outside [0, 1], so it is not a discount fraction and its cost basis is unknow...


## 4 · The joint solve — promo competes for the same money

`build_arm_curves` puts every arm on one **realized-cost** grid: media arms at
their spend, the promo arm at `depth × unit_cost`. The budget constraint is
already `Σ cost = B`, so the whole existing machinery — risk objectives,
constraints, the multi-start solver, per-draw decision uncertainty, the
out-of-support flag — runs unchanged, and the media-only path stays
bit-identical (pinned by test). New: a **concavity gate** — the greedy
allocator's exactness precondition is finally checked, and a failing arm
forces the multi-start constrained solver with a note.

In [6]:
from mmm_framework.planning import build_arm_curves, optimize_arms

t0 = time.time()
curves = build_arm_curves(mmm, promo_var="Promo", unit_cost=t["promo_unit_cost"],
                          max_draws=150, random_seed=42)
print(f"arm curves in {time.time()-t0:.0f}s")
for a in curves.arms:
    print(f"  {a.name:22s} kind={a.kind:5s} units={a.level_units}")

budget = t["true_optimal_split"]["budget"]
res = optimize_arms(curves=curves, total_budget=budget,
                    value_per_kpi=value_per_kpi, value_source="world economics",
                    min_multiplier=0.3, max_multiplier=1.5, random_seed=42)
cols = ["channel", "arm_kind", "optimal_spend", "optimal_level", "level_units",
        "change_pct", "within_observed_range"]
print("\n" + res.table[cols].round(3).to_string(index=False))

arm curves in 10s
  TV                     kind=media units=$
  Search                 kind=media units=$
  Social                 kind=media units=$
  Display                kind=media units=$
  Promo depth (Promo)    kind=promo units=avg weekly depth (fraction)



            channel arm_kind  optimal_spend  optimal_level                 level_units  change_pct  within_observed_range
                 TV    media       7929.960       7929.960                           $     -12.919                   True
             Search    media       4651.598       4651.598                           $     -24.517                   True
             Social    media       4641.509       4641.509                           $       0.902                   True
            Display    media       3796.594       3796.594                           $      -0.667                   True
Promo depth (Promo)    promo       8100.915          0.068 avg weekly depth (fraction)      49.200                   True


The promo row's `optimal_spend` is **margin given away**, not a media buy —
which is why the table carries `arm_kind` and reports the recommendation in
the arm's own units (`optimal_level`, an average weekly depth).

In [7]:
promo_row = res.table[res.table.arm_kind == "promo"].iloc[0]
rec_share = float(promo_row["optimal_spend"]) / budget
cur_share = t["current_split"]["promo_cost"] / budget
true_share = t["true_optimal_split"]["promo_share"]

fig = go.Figure(go.Bar(
    x=["current", "recommended", "planted optimum"],
    y=[cur_share, rec_share, true_share],
    marker_color=[MUTED, PALETTE["Promo depth (Promo)"], TRUTH],
))
fig.update_yaxes(title="promo share of total outlay", tickformat=".0%")
style(fig, 340, "The recommendation moves toward the planted optimum")
fig.show()

# TRUE planted profit of the recommendation (noiseless structural mean, same
# frozen economics the answer key used — conditional on them, as the world's
# own notes insist).
fn = t["lever_response_fn"]; n = len(scenario.y)
spend = scenario.spend.to_numpy(float)
alloc = {r_["channel"]: r_["optimal_spend"] for _, r_ in res.table.iterrows()
         if r_["arm_kind"] == "media"}
scale = np.array([alloc[c] / spend[:, i].sum() for i, c in enumerate(scenario.channels)])
d_rec = float(promo_row["optimal_level"])
mu_rec = fn(spend * scale[None, :], np.full(n, d_rec), t["price"]).sum()
profit_rec = value_per_kpi * mu_rec - (sum(alloc.values()) + d_rec * t["promo_unit_cost"])

print(f"recommended promo share : {rec_share:.1%}  (current {cur_share:.1%}, planted optimum {true_share:.1%})")
print(f"TRUE profit — current   : {t['current_split']['profit']:,.0f}")
print(f"TRUE profit — recommended: {profit_rec:,.0f}")
print(f"TRUE profit — planted opt: {t['true_optimal_split']['profit']:,.0f}")
print(f"decision regret vs optimum: {t['true_optimal_split']['profit'] - profit_rec:,.0f}")

recommended promo share : 27.8%  (current 18.6%, planted optimum 55.3%)
TRUE profit — current   : 456,246
TRUE profit — recommended: 465,067
TRUE profit — planted opt: 480,533
decision regret vs optimum: 15,466


The recommendation moves decisively toward the planted optimum. It does not
reach it — the fitted promo lift recovers ~85% of truth and the fitted media
saturation is imperfect — and the honest grade is the **decision regret**
against the planted optimum, printed above, not a victory lap.

## 5 · Price: evaluate, label, refuse

The price lever's elasticity is measured at ~39% of planted truth by the
repo's own published simulation of this exact mechanism. So the price surface
answers stated hypotheticals and **refuses to emit a recommendation** — a
Granger-style screen not flagging is weak evidence of exogeneity, and the
default posture is refusal regardless of the flag.

In [8]:
from mmm_framework.planning import price_whatif

w = price_whatif(mmm, 0.95, max_draws=150)
print(f"scenario: {w['price_var']} x {w['factor']} (a 5% cut)")
print(f"KPI delta: {w['kpi_delta_mean']:,.0f} "
      f"[{w['kpi_delta_lower']:,.0f}, {w['kpi_delta_upper']:,.0f}]")
print(f"recommendation: {w['recommendation']}")
print(f"\nwhy it refuses:\n  {w['refusal_reason']}")

scenario: Price x 0.95 (a 5% cut)
KPI delta: 541 [69, 1,149]
recommendation: None

why it refuses:
  Price recommendations are refused by design: the shipped sign-guarded elasticity recovers ~39% of a planted truth in the repo's own measurement (9% with no designed price variation), so the P&L consequence of a recommended move would be ~2.5x what the model believes. This scenario is an evaluation of a stated hypothetical, conditional on the fitted (attenuated) elasticity — run a pricing experiment before acting.


## 6 · The negative control — clearance timing

In `promo_endogenous`, last week's soft demand triggers this week's deeper
promo and price cut. The lever now correlates with the error term: the naive
lift is **attenuated by construction**, and the extended endogeneity screen —
which now walks lever columns with a `kind` — shows the demand→setting lead.

In [9]:
from mmm_framework.diagnostics.endogeneity import endogeneity_diagnostic

sc_en = dgp.build("promo_endogenous")
DATA_EN = ART / "promo_endo.csv"
mff.scenario_to_mff(sc_en).to_csv(DATA_EN, index=False)
spec_en = dict(SPEC)
spec_en["control_variables"] = [{"name": c} for c in sc_en.controls.columns]
m_en = build_model(spec_en, str(DATA_EN))
m_en.fit(draws=800, tune=800, chains=4, random_seed=42)

d = endogeneity_diagnostic(m_en)
rows = [{"column": r_["channel"], "kind": r_["kind"],
         "demand→setting": round(r_["demand_leads_spend"], 3),
         "setting→demand": round(r_["spend_leads_demand"], 3),
         "flagged": r_["endogenous"]} for r_ in d["channels"]]
print(pd.DataFrame(rows).to_string(index=False))
print("\nflagged levers:", d["flagged_levers"])

r_en = promo_roi(m_en, "Promo", unit_cost=sc_en.notes["promo_unit_cost"],
                 value_per_kpi=1.0)
r_ex = promo_roi(mmm, "Promo", unit_cost=t["promo_unit_cost"], value_per_kpi=1.0)
print(f"\nlift recovery — exogenous timing : {r_ex.lift_kpi_mean/t['true_promo_lift']:.0%}")
print(f"lift recovery — clearance timing : {r_en.lift_kpi_mean/sc_en.notes['true_promo_lift']:.0%}")

  0%|          | 0/1600 [00:00<?, ?it/s]

  0%|          | 0/1600 [00:00<?, ?it/s]

  0%|          | 0/1600 [00:00<?, ?it/s]

  0%|          | 0/1600 [00:00<?, ?it/s]

 column  kind  demand→setting  setting→demand  flagged
     TV media          -0.202          -0.269    False
 Search media           0.206          -0.215    False
 Social media          -0.153          -0.120    False
Display media          -0.138           0.161    False
  Price price           0.364          -0.102     True
  Promo promo          -0.276           0.353    False

flagged levers: ['Price']



lift recovery — exogenous timing : 89%
lift recovery — clearance timing : 70%


## Where this surfaces

- **Planner tables** — allocation rows carry `arm_kind` / `level_units` /
  `optimal_level`; the frontend shows Kind and Level columns whenever a
  non-media arm is present.
- **Refusals upstream** — `compute_response_curves` now refuses a channel
  whose divisor is not monetary (impressions summed into a dollar budget), and
  `goal_seek` refuses a mixed-arm portfolio by name (its monotone-frontier
  proof covers concave spend curves only).
- **The valuation chain** — the agent tool's `value_per_kpi` default changed
  from a silent `1.0` to `None`: fund-to-breakeven now refuses without a
  declared valuation instead of asserting one KPI unit = one dollar.

### Reading list

- `nbs/demos/payback_horizon.ipynb` — the same epistemics discipline for
  response timing.
- `docs/blog-modelled-one-p.html` — why the price arm refuses: the 39%
  measurement.
- `tests/test_decision_arms.py` — the analytic equal-marginal-profit gate, the
  10-seed milestone, and the attenuation floor.